In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import display, HTML
import sqlite3
import mysql.connector
import pyarrow
import argparse


def extract_data(exchange):
    
    #exchange = "BTC-USD"
    get_date = datetime.now()
    get_date_formated = get_date.strftime("%Y-%m-%d")
    get_date_730_days_ago = get_date - timedelta(days=729)
    get_date_730_days_ago_formated = get_date_730_days_ago.strftime("%Y-%m-%d")

    daily_data = yf.download(exchange
                    ,start = "2012-01-01"
                    ,end = get_date_formated)



    hour_data = yf.download(exchange
                    ,start = get_date_730_days_ago_formated
                    ,end = get_date_formated
                    ,interval = "1h")
    return daily_data, hour_data


def transform_daily(dataframe, exchange):
    # Add the exchange column
    dataframe['exchange'] = exchange

    #Reset index to extract date column
    dataframe.reset_index(inplace=True)

    #Extract the date part of the datetime column
    # We are converting the Date to id_date format
    # Example: 2024-04-17 is converted to 20240417

    dataframe['Date'] = dataframe['Date'].astype(str)
    dataframe['Date'] = dataframe['Date'].str.replace('-', '')
    return dataframe
    
def transform_hour(dataframe, exchange):
    # Add the exchange column
    dataframe['exchange'] = exchange

    # Reset index to extract date column
    dataframe.reset_index(inplace=True)

    # Separe the date and time parts
    dataframe['Datetime'] = dataframe['Datetime'].astype(str)
    dataframe[['Date', 'Hour']] = dataframe['Datetime'].str.split(' ', expand=True)

    # Extract date and hour data
    # We are converting the Date to id_date format
    # Example: 2024-04-17 is converted to 20240417

    dataframe['Date'] = dataframe['Date'].str.replace('-', '')
    dataframe['Hour'] = dataframe['Hour'].str.split(':', expand=True)[0]
    dataframe['Hour'] = dataframe['Hour'].astype(int)

    #Drop datetime data
    dataframe = dataframe.drop(columns = ['Datetime'])
    return dataframe




In [2]:
def get_id_exchange():
    connection = mysql.connector.connect(
        user = 'root',
        password = 'root',
        host = 'localhost',
        port = 3306,
        database = 'Historical_Data'
    )
    print("MySQL DB Connected")

    cursor = connection.cursor()

    cursor.execute("SELECT * FROM DT_EXCHANGES")

    results = cursor.fetchall()


    columns = [column[0] for column in cursor.description]


    df_dt_exchanges = pd.DataFrame(results, columns=columns)


    cursor.close()
    connection.close()

    df_dt_exchanges = df_dt_exchanges[['exchange', 'id_exchange']]
    return df_dt_exchanges



def load_daily_data(dataframe):
    connection = mysql.connector.connect(
    user = 'root',
    password = 'root',
    host = 'localhost',
    port = 3306,
    database = 'Historical_Data'
    )
    print("MySQL DB Connected")

    cursor = connection.cursor()
    
    #Truncate table
    exchange = dataframe['exchange'].unique()[0]
    
    cursor.execute(f"""DELETE FROM FT_DAILY_DATA WHERE Exchange = '{exchange}'""")
    
    
    #Insert data
    cursor.execute("""SET FOREIGN_KEY_CHECKS = 0""")


    # SQL Consult
    sql_insert = """
        INSERT INTO FT_DAILY_DATA (id_date, Open, High, Low, Close, adj_close, Volume, Exchange, id_exchange)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    try:
        # Iterate on the dataframe
        for index, row in dataframe.iterrows():
            cursor.execute(sql_insert, tuple(row))
        
        # Confirm the changes on the database
        connection.commit()
        print("Data inserted correctly on the DAILY_DATA table.")
    except mysql.connector.Error as error:
        # Error
        print("Error inserting the DAILY data:", error)
        connection.rollback()
        
    cursor.execute("""SET FOREIGN_KEY_CHECKS = 1""")

    # Close the cursor and the connection
    cursor.close()
    connection.close()
    
    
    
def load_hour_data(dataframe):
    connection = mysql.connector.connect(
        user = 'root',
        password = 'root',
        host = 'localhost',
        port = 3306,
        database = 'Historical_Data'
    )
    print("MySQL DB Connected")
    
    

    cursor = connection.cursor()
    
    #Truncate table
    exchange = dataframe['exchange'].unique()[0]
    
    cursor.execute(f"""DELETE FROM FT_HOUR_DATA WHERE Exchange = '{exchange}'""")
    
    #Insert data

    cursor.execute("""SET FOREIGN_KEY_CHECKS = 0""")


    # SQL Consult
    sql_insert = """
        INSERT INTO FT_HOUR_DATA (Open, High, Low, Close, adj_close, Volume, Exchange, id_date, Hour, id_exchange)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    try:
        # Iterate on the dataframe
        for index, row in dataframe.iterrows():
            cursor.execute(sql_insert, tuple(row))
        
        # Confirm the changes on the database
        connection.commit()
        print("Data inserted correctly on the HOUR_DATA table.")
    except mysql.connector.Error as error:
        # Error
        print("Error inserting the HOUR data:", error)
        connection.rollback()
        
    cursor.execute("""SET FOREIGN_KEY_CHECKS = 1""")

    # Close the cursor and the connection
    cursor.close()
    connection.close()
    




In [9]:
df_dt_exchanges

,exchange,id_exchange
0,BTC-USD,96
1,ETH-USD,97
2,XRP-USD,98
3,LTC-USD,99
4,BCH-USD,100
5,LINK-USD,101
6,ADA-USD,102
7,DOT-USD,103
8,XLM-USD,104
9,USDT-USD,105


In [8]:
exchange = '^GSPC'
    
daily_data, hour_data = extract_data(exchange)
daily_data = transform_daily(daily_data, exchange)
hour_data = transform_hour(hour_data, exchange)

df_dt_exchanges = get_id_exchange()

daily_data = pd.merge(daily_data, df_dt_exchanges,
                    on='exchange', how='left')

daily_data

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


MySQL DB Connected


,Date,Open,High,Low,Close,Adj Close,Volume,exchange,id_exchange
0,20120103,1258.859985,1284.619995,1258.859985,1277.060059,1277.060059,3943710000,^GSPC,NaN
1,20120104,1277.030029,1278.729980,1268.099976,1277.300049,1277.300049,3592580000,^GSPC,NaN
2,20120105,1277.300049,1283.050049,1265.260010,1281.060059,1281.060059,4315950000,^GSPC,NaN
3,20120106,1280.930054,1281.839966,1273.339966,1277.810059,1277.810059,3656830000,^GSPC,NaN
4,20120109,1277.829956,1281.989990,1274.550049,1280.699951,1280.699951,3371600000,^GSPC,NaN
...,...,...,...,...,...,...,...,...,...
3141,20240628,5488.479980,5523.640137,5451.120117,5460.479980,5460.479980,7199220000,^GSPC,NaN
3142,20240701,5471.080078,5479.549805,5446.529785,5475.089844,5475.089844,3488760000,^GSPC,NaN
3143,20240702,5461.839844,5509.689941,5458.430176,5509.009766,5509.009766,3329950000,^GSPC,NaN
3144,20240703,5507.439941,5539.270020,5507.419922,5537.020020,5537.020020,2179470000,^GSPC,NaN


In [ ]:
def load_data(exchange):
    
    daily_data, hour_data = extract_data(exchange)

    daily_data = transform_daily(daily_data, exchange)
    hour_data = transform_hour(hour_data, exchange)

    df_dt_exchanges = get_id_exchange()

    daily_data = pd.merge(daily_data, df_dt_exchanges,
                        on='exchange', how='left')

    hour_data = pd.merge(hour_data, df_dt_exchanges,
                        on = 'exchange', how = 'left')

    load_daily_data(daily_data)

    load_hour_data(hour_data)
    


    
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Load historical data for a given exchange.")
    parser.add_argument("exchange", type=str, help="The exchange symbol to download data for.")
    args = parser.parse_args()
    load_data(args.exchange)